In [1]:
import sys
import pandas as pd
import numpy as np
import IPython
from IPython.display import display

print( f"Python {sys.version}" )
print( f"Pandas {pd.__version__}" )
print( f"NumPy {np.__version__}" )
print( f"IPython {IPython.__version__}" )

Python 3.13.14 (main, Jul 14 2026, 04:45:36) [GCC 14.2.0]
Pandas 3.0.5
NumPy 2.5.1
IPython 9.16.1


In [2]:
# Load the pokemon dataset
df_pokemon = pd.read_csv('https://raw.githubusercontent.com/ShaileshDhama/Exploratory-Data-Analysis-On-Pokemon-Dataset/master/Complete%20Pokemon.csv')
df_pokemon.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   abilities          801 non-null    str    
 1   against_bug        801 non-null    float64
 2   against_dark       801 non-null    float64
 3   against_dragon     801 non-null    float64
 4   against_electric   801 non-null    float64
 5   against_fairy      801 non-null    float64
 6   against_fight      801 non-null    float64
 7   against_fire       801 non-null    float64
 8   against_flying     801 non-null    float64
 9   against_ghost      801 non-null    float64
 10  against_grass      801 non-null    float64
 11  against_ground     801 non-null    float64
 12  against_ice        801 non-null    float64
 13  against_normal     801 non-null    float64
 14  against_poison     801 non-null    float64
 15  against_psychic    801 non-null    float64
 16  against_rock       801 non-null    fl

# 3. Pandas: Groupby

Pandas’ `groupby()` allows us to split data into separate groups to perform computations for better analysis.
- Efficient data summarization
- Allow us to squeeze data (This is very useful when the data is so big and cannot fit in one Series/DataFrame)



## 3.1 The groupby object is neither pandas.Series nor pandas.DataFrame

Groupby on pandas.Series:

- When `groupby()` is used on Series, our job is to assign a group label to each row in the Series.

>ผลลัพธ์ของการ groupby ไม่ใช่ทั้ง series และ dataframe

In [3]:
# Create a pandas.Series to play around
ser = pd.Series([390., 350., 30., 20.])
ser

0    390.0
1    350.0
2     30.0
3     20.0
dtype: float64

In [ ]:
# Determine a group for each Series' element by a list of group labels

# In the below command, the 0th and 2nd rows are assigned to the "a" group and the other two to the "b" group
grouped = ser.groupby( ["a", "b", "a", "b"] )

# The groupby object is neither Series nor DataFrame, be careful when dealing with it
grouped

# สังเกตว่า data ที่ได้จะเป็น groupby object ไม่ใช้ series หรือ dataframe แล้ว (บางตำสั่งจะใช้ไม่ได้)

In [ ]:
# Inspect the result of the above cell by iterating through the groupby object
# เราสามาถ ใช้ for loop กับ groupby object ได้
for i, (name,group) in enumerate(grouped):
  print( f"===== Group {i} =====" )
  print( f"Group name: {name}" )
  print( f"Type: {type(group)}" )
  print( f"Value:\n{group}\n" )

===== Group 0 =====
Group name: a
Type: <class 'pandas.Series'>
Value:
0    390.0
2     30.0
dtype: float64

===== Group 1 =====
Group name: b
Type: <class 'pandas.Series'>
Value:
1    350.0
3     20.0
dtype: float64



In [ ]:
# Use a function to specify how to group each element in the pandas.Series
# In the lambda function below:
# - "i" refers to a label of a row
# - Any element (in a Series) whose value is greater than 300, will be assigned to the "A" group or the "B" group otherwise
# เราสามารถ lambda function ในการเขียน condition ได้ 
# ตัวอย่าง check label ของ row
grouped = ser.groupby( lambda i: "A" if ser.loc[i]>300 else "B" )

# Inspect the result by iterating through the groupby object
for i, (name,group) in enumerate(grouped):
  print( f"===== Group {i} =====" )
  print( f"Group name: {name}" )
  print( f"Type: {type(group)}" )
  print( f"Value:\n{group}\n" )

===== Group 0 =====
Group name: A
Type: <class 'pandas.Series'>
Value:
0    390.0
1    350.0
dtype: float64

===== Group 1 =====
Group name: B
Type: <class 'pandas.Series'>
Value:
2    30.0
3    20.0
dtype: float64



Groupby on pandas.DataFrame:

In [7]:
# Prepare a pandas.DataFrame to play around
df = df_pokemon.loc[ :, ['pokedex_number','name','type1','type2'] ]

In [ ]:
# By default, any group with NA key will be dropped. To include NA-key group(s), set dropna=False.
# BUT having any group with NA key will lead to future errors (e.g., .groups).
# One suggestion is to use fillna() to replace NA keys before doing groupby().

# Compare groupby with 'dropna=True' versus 'dropna=False'
# ตัวอย่างนี้ groupby โดยใช้ column 'type2' ซึ่งมี missing value อยู่
grouped = df.groupby( ['type2'] )     # default: axis=0, dropna=True
grouped_na = df.groupby( ['type2'], dropna=False )

# The groupby object is neither Series nor DataFrame, be careful when dealing with it
# ไม่สามารถ display ออกมาให้ดูได้ เนื่องจาก groupby object ไม่ใช่ทั้ง series และ dataframe
# เราไม่สามารถใช้ทำเหมือน series และ dataframe ได้
display(grouped)
display(grouped_na)

In [12]:
# Access attribute of the groupby object
# groupby มีการเก็บค่า parameter สำคัญๆให้เราสามาถเรียกใช้ได้เลย

# Get the number of groups
print(f'{grouped.ngroups=}')      # OK
print(f'{grouped_na.ngroups=}')   # OK

grouped.ngroups=18
grouped_na.ngroups=19


In [13]:
# Access attribute of the groupby object

# Get the group objects
# Each resultant list contains integer indices of rows that belong to the group
# Preview ข้อมูลของแต่ละกลุ่ม return ออกมาเป็น dictionary ซึ่ง key คือ ชื่อ 'type2' value row
display(grouped.groups)      # OK
display(grouped_na.groups)   # ValueError because of the key=null category

/tmp/ipykernel_43107/3377070693.py:6: Pandas4Warning: In a future version, the keys of `groups` will be a tuple with a single element, e.g. (type2,) , instead of a scalar, e.g. type2, when grouping by a list with a single element. Use ``df.groupby(by='a').groups`` instead of ``df.groupby(by=['a']).groups`` to avoid this warning
  display(grouped.groups)      # OK


{'bug': [346, 347, 450, 750, 751], 'dark': [18, 19, 51, 52, 247, 273, 274, 317, 318, 331, 341, 433, 434, 441, 451, 550, 551, 552, 657, 674, 726], 'dragon': [229, 328, 329, 482, 483, 486, 632, 633, 634, 690, 695, 696, 713, 714, 775, 779, 798], 'electric': [25, 169, 170, 594, 595, 617, 643, 736, 737], 'fairy': [38, 39, 121, 173, 182, 183, 279, 280, 281, 297, 302, 438, 545, 546, 701, 702, 706, 718, 729, 741, 742, 754, 755, 777, 784, 785, 786, 787, 800], 'fighting': [61, 213, 255, 256, 285, 390, 391, 452, 453, 474, 498, 499, 558, 559, 637, 638, 639, 646, 651, 758, 759, 782, 783, 793, 794], 'fire': [104, 227, 228, 493, 554, 606, 607, 608, 635, 636, 642, 756, 757], 'flying': [5, 11, 15, 16, 17, 20, 21, 40, 41, 82, 83, 84, 122, 129, 141, 143, 144, 145, 148, 162, 163, 164, 165, 168, 175, 176, 177, 186, 187, 188, 192, 197, 206, 224, 225, 226, 248, 249, 266, 275, 276, 277, 278, 283, 290, 332, 333, 356, 372, 383, 395, 396, 397, 413, 414, 415, 424, 425, 429, 440, 457, 467, 468, 471, 518, 519, 520,

/tmp/ipykernel_43107/3377070693.py:7: Pandas4Warning: In a future version, the keys of `groups` will be a tuple with a single element, e.g. (type2,) , instead of a scalar, e.g. type2, when grouping by a list with a single element. Use ``df.groupby(by='a').groups`` instead of ``df.groupby(by=['a']).groups`` to avoid this warning
  display(grouped_na.groups)   # ValueError because of the key=null category


{'bug': [346, 347, 450, 750, 751], 'dark': [18, 19, 51, 52, 247, 273, 274, 317, 318, 331, 341, 433, 434, 441, 451, 550, 551, 552, 657, 674, 726], 'dragon': [229, 328, 329, 482, 483, 486, 632, 633, 634, 690, 695, 696, 713, 714, 775, 779, 798], 'electric': [25, 169, 170, 594, 595, 617, 643, 736, 737], 'fairy': [38, 39, 121, 173, 182, 183, 279, 280, 281, 297, 302, 438, 545, 546, 701, 702, 706, 718, 729, 741, 742, 754, 755, 777, 784, 785, 786, 787, 800], 'fighting': [61, 213, 255, 256, 285, 390, 391, 452, 453, 474, 498, 499, 558, 559, 637, 638, 639, 646, 651, 758, 759, 782, 783, 793, 794], 'fire': [104, 227, 228, 493, 554, 606, 607, 608, 635, 636, 642, 756, 757], 'flying': [5, 11, 15, 16, 17, 20, 21, 40, 41, 82, 83, 84, 122, 129, 141, 143, 144, 145, 148, 162, 163, 164, 165, 168, 175, 176, 177, 186, 187, 188, 192, 197, 206, 224, 225, 226, 248, 249, 266, 275, 276, 277, 278, 283, 290, 332, 333, 356, 372, 383, 395, 396, 397, 413, 414, 415, 424, 425, 429, 440, 457, 467, 468, 471, 518, 519, 520,

In [ ]:
# Access method of the groupby object

# แจกแจงออกมาว่าค่าในกลุ่ม มีจำนวนเท่าไหร่
# Compute and display group sizes
print( grouped.size(), end='\n\n' ) # OK
print( grouped_na.size() )          # OK

type2
bug          5
dark        21
dragon      17
electric     9
fairy       29
fighting    25
fire        13
flying      95
ghost       14
grass       20
ground      34
ice         15
normal       4
poison      34
psychic     29
rock        14
steel       22
water       17
dtype: int64

type2
bug           5
dark         21
dragon       17
electric      9
fairy        29
fighting     25
fire         13
flying       95
ghost        14
grass        20
ground       34
ice          15
normal        4
poison       34
psychic      29
rock         14
steel        22
water        17
NaN         384
dtype: int64


In [ ]:
# Access method of the groupby object

# Preview groups (and the first entry in each group) as a pandas.DataFrame
# preview member ตัวแรกของกลุ่มขึ้นมาให้ดู
display( grouped.first() )      # OK
print()
display( grouped_na.first() )   # OK

,pokedex_number,name,type1
type2,,,
bug,347,Anorith,rock
dark,19,Rattata,normal
dragon,230,Kingdra,water
electric,26,Raichu,electric
fairy,39,Jigglypuff,normal
fighting,62,Poliwrath,water
fire,105,Marowak,ground
flying,6,Charizard,fire
ghost,292,Shedinja,bug


,pokedex_number,name,type1
type2,,,
bug,347,Anorith,rock
dark,19,Rattata,normal
dragon,230,Kingdra,water
electric,26,Raichu,electric
fairy,39,Jigglypuff,normal
fighting,62,Poliwrath,water
fire,105,Marowak,ground
flying,6,Charizard,fire
ghost,292,Shedinja,bug


In [16]:
# Access method of the groupby object

# Preview groups (and the last entry in each group) as a pandas.DataFrame
display( grouped.last() )      # OK
print()
display( grouped_na.last() )   # OK

,pokedex_number,name,type1
type2,,,
bug,752,Araquanid,water
dark,727,Incineroar,fire
dragon,799,Guzzlord,dark
electric,738,Vikavolt,bug
fairy,801,Magearna,steel
fighting,795,Pheromosa,bug
fire,758,Salazzle,poison
flying,797,Celesteela,steel
ghost,792,Lunala,psychic


,pokedex_number,name,type1
type2,,,
bug,752,Araquanid,water
dark,727,Incineroar,fire
dragon,799,Guzzlord,dark
electric,738,Vikavolt,bug
fairy,801,Magearna,steel
fighting,795,Pheromosa,bug
fire,758,Salazzle,poison
flying,797,Celesteela,steel
ghost,792,Lunala,psychic


In [ ]:
# Access method of the groupby object

# ในกรณีที่เราอยาก  Access กลุ่มใดกลุ่มหนึ่งเป็นพิเศษ เราสามารถระบุ index ชื่อของกลุ่ม ออกมาได้เลย
# NEW codes: Return a pandas.DataFrame of a specified group
display( df.iloc[ grouped.indices.get('dragon') ] ) # OK
display( df.iloc[ grouped_na.indices.get(pd.NA) ] ) # TypeError: Cannot index by location index with a non-integer key

# OLD codes: grouped.get_group() method is deprecated since version 2.1.0
#display( grouped.get_group(('dragon',)) )   # OK
#display( grouped_na.get_group(pd.NA) )      # Error: it doesn't know how to access the NA-key group

,pokedex_number,name,type1,type2
229,230,Kingdra,water,dragon
328,329,Vibrava,ground,dragon
329,330,Flygon,ground,dragon
482,483,Dialga,steel,dragon
483,484,Palkia,water,dragon
486,487,Giratina,ghost,dragon
632,633,Deino,dark,dragon
633,634,Zweilous,dark,dragon
634,635,Hydreigon,dark,dragon
690,691,Dragalge,poison,dragon


TypeError: Cannot index by location index with a non-integer key

In [ ]:
# Iterate through groups

# เราสามารถ forloop ดูแต่ละ member ได้เหมือนเดิม
# Below codes work fine even for NA-key groups
for i, (name, group) in enumerate(grouped):
  if i == 3:    # As a demo, we inspect only the first three groups
    break
  print( f"\n===== Group {i} =====" )
  print( f"Group name: type={type(name)} value={name}" )
  print( f"Group: type={type(group)}" )
  display(group)


===== Group 0 =====
Group name: type=<class 'tuple'> value=('bug',)
Group: type=<class 'pandas.DataFrame'>


,pokedex_number,name,type1,type2
346,347,Anorith,rock,bug
347,348,Armaldo,rock,bug
450,451,Skorupi,poison,bug
750,751,Dewpider,water,bug
751,752,Araquanid,water,bug



===== Group 1 =====
Group name: type=<class 'tuple'> value=('dark',)
Group: type=<class 'pandas.DataFrame'>


,pokedex_number,name,type1,type2
18,19,Rattata,normal,dark
19,20,Raticate,normal,dark
51,52,Meowth,normal,dark
52,53,Persian,normal,dark
247,248,Tyranitar,rock,dark
273,274,Nuzleaf,grass,dark
274,275,Shiftry,grass,dark
317,318,Carvanha,water,dark
318,319,Sharpedo,water,dark
331,332,Cacturne,grass,dark



===== Group 2 =====
Group name: type=<class 'tuple'> value=('dragon',)
Group: type=<class 'pandas.DataFrame'>


,pokedex_number,name,type1,type2
229,230,Kingdra,water,dragon
328,329,Vibrava,ground,dragon
329,330,Flygon,ground,dragon
482,483,Dialga,steel,dragon
483,484,Palkia,water,dragon
486,487,Giratina,ghost,dragon
632,633,Deino,dark,dragon
633,634,Zweilous,dark,dragon
634,635,Hydreigon,dark,dragon
690,691,Dragalge,poison,dragon


In [ ]:
# Group by many columns
# Groupby มากกว่า 1 column เหมือน sql
# ตัวอย่าง groupby โดยใช้ 2 column 
group2 = df.groupby( ['type1', 'type2'] )

# The results include groups with type2=nan (e.g., ('dragon', nan), ('fighting', nan))
print(f'{group2.ngroups=}\n')
display(group2.groups)

group2.ngroups=148



{('bug', 'electric'): Index([594, 595, 736, 737], dtype='int64'),
 ('bug', 'fairy'): RangeIndex(start=741, stop=743, step=1),
 ('bug', 'fighting'): Index([213, 793, 794], dtype='int64'),
 ('bug', 'fire'): RangeIndex(start=635, stop=637, step=1),
 ('bug',
  'flying'): Index([11, 122, 164, 165, 192, 266, 283, 290, 413, 414, 415, 468, 665], dtype='int64'),
 ('bug', 'ghost'): RangeIndex(start=291, stop=292, step=1),
 ('bug', 'grass'): Index([45, 46, 412, 539, 540, 541], dtype='int64'),
 ('bug', 'ground'): RangeIndex(start=289, stop=290, step=1),
 ('bug',
  'poison'): Index([12, 13, 14, 47, 48, 166, 167, 268, 542, 543, 544], dtype='int64'),
 ('bug', 'rock'): Index([212, 556, 557], dtype='int64'),
 ('bug', 'steel'): Index([204, 211, 588, 631, 648], dtype='int64'),
 ('bug', 'water'): Index([282, 766, 767], dtype='int64'),
 ('dark', 'dragon'): Index([632, 633, 634, 798], dtype='int64'),
 ('dark', 'fighting'): RangeIndex(start=558, stop=560, step=1),
 ('dark', 'fire'): RangeIndex(start=227, sto

In [ ]:
# NEW codes: When grouping is done based on many columns, use a tuple to specify the group name
df.iloc[ group2.indices.get(('bug','poison')) ]

# OLD codes: grouped.get_group() method is deprecated since version 2.1.0
# ถ้าเรา groupby โดยใช้มากกว่า 1 columns ต้อง access โดยใช้ชื่อของทั้ง / กลุ่ม อยู่ในรูปแบบ tuple
group2.get_group( ('bug','poison') )

,pokedex_number,name,type1,type2
12,13,Weedle,bug,poison
13,14,Kakuna,bug,poison
14,15,Beedrill,bug,poison
47,48,Venonat,bug,poison
48,49,Venomoth,bug,poison
166,167,Spinarak,bug,poison
167,168,Ariados,bug,poison
268,269,Dustox,bug,poison
542,543,Venipede,bug,poison
543,544,Whirlipede,bug,poison


## 3.2 The groupby process: split-apply-combine

According to https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html, “group by” refers to a process involving one or more of the following steps:
- Splitting the data into groups based on some criteria
- **Applying a function to each group independently** (e.g., aggregation, transformation, filtration)
- Combining the results into a data structure

## 3.2 The groupby process: **split-apply-combine**

According to https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html, “group by” refers to a process involving one or more of the following steps:
- Splitting the data into groups based on some criteria
- **Applying a function to each group independently** (e.g., aggregation, transformation, filtration)
- Combining the results into a data structure

### 3.2.1 Split $→$ Aggregate<a name='sect3-2-1'>

![picture](https://miro.medium.com/max/826/1*WTlC_t-5RQqeqS0tcbkCKQ.png)

Image from <a href="https://iamirmasoud.com/2021/06/26/a-complete-guide-on-pandas-grouping-aggregating-and-transformation/">https://iamirmasoud.com/2021/06/26/a-complete-guide-on-pandas-grouping-aggregating-and-transformation/</a>

In [22]:
# Prepare a groupby object to play around
group = df_pokemon[['pokedex_number','name','type1','type2','hp','attack','defense']].groupby('type1')
group

In [24]:
# Choose some existing columns from the groupby object

# Although it looks like choosing columns in pandas.DataFrame, the result below remains the groupby object
group[['hp','attack']] 
# groupby object ไม่มีคำสั่ง .loc           # OK

group.loc[ :, ['hp','attack'] ]  # Error: Groupby has no attribute 'loc'

AttributeError: 'DataFrameGroupBy' object has no attribute 'loc'

In [ ]:
# Group -> Choose some existing columns -> Apply the aggregation function
# For each group, apply an aggregation function to each column separately

# As 'mean' is numeric aggregation, non-numeric columns (i.e., 'name','type1','type2') are excluded (with warning)
# syntax นี้ สามารถใช้กับ ในกรณีที่ ใส้ในเป็น dataframe
# ได้ผลลัพธ์ที่ apply aggregate แยกกันภายในกลุ่ม
group[['hp','attack','defense']].agg('mean')   # This equals group[['hp','attack','defense']].mean() as well as group[['hp','attack','defense']].agg(np.mean)

#group.agg('mean')     # TypeError

,hp,attack,defense
type1,,,
bug,56.722222,70.125000,70.847222
dark,72.551724,87.793103,70.517241
dragon,79.851852,106.407407,86.259259
electric,60.512821,70.820513,61.820513
fairy,73.944444,62.111111,68.166667
fighting,71.428571,99.178571,66.392857
fire,68.730769,81.500000,67.788462
flying,68.000000,66.666667,65.000000
ghost,63.370370,72.740741,79.518519


In [ ]:
# Group -> Choose some existing columns -> Apply the aggregation functions
result = group[['hp','attack']].agg(['mean','median','min','max','std'])

# Result is a DataFrame with two-level column indices
result
# ถ้าเรา groupby แล้วเอาไป aggregate ผลลัพธ์ที่ได้ออกมาจะกลับมาเป็น  dataframe

hp                                 attack                  \
               mean median min  max        std        mean median min  max   
type1                                                                        
bug       56.722222   60.0   1  107  17.652532   70.125000   65.0  10  185   
dark      72.551724   65.0  35  223  36.063026   87.793103   88.0  50  150   
dragon    79.851852   75.0  41  216  34.962291  106.407407  100.0  50  180   
electric  60.512821   60.0  20   90  17.519721   70.820513   65.0  30  123   
fairy     73.944444   76.0  35  126  23.048471   62.111111   58.5  20  131   
fighting  71.428571   70.0  30  144  26.790585   99.178571  100.0  35  145   
fire      68.730769   65.0  38  115  19.182422   81.500000   79.5  30  160   
flying    68.000000   79.0  40   85  24.433583   66.666667   70.0  30  100   
ghost     63.370370   59.0  20  150  30.011726   72.740741   66.0  30  165   
grass     65.358974   65.0  30  123  18.721454   73.769231   68.0  27  181   
ground    73.187500   72.5  10  115  25.518415   94.812500   90.0  40  180   
ice       72.086957   70.0  36  110  21.760918   73.304348   65.0  30  130   
normal    76.723810   70.0  30  255  35.460838   75.161905   75.0   5  160   
poison    65.593750   65.0  35  105  19.108956   72.656250   68.5  43  106   
psychic   72.943396   68.0  20  190  31.507884   65.566038   55.0  20  165   
rock      66.333333   67.0  30  123  21.781978   90.666667   89.0  40  165   
steel     66.791667   60.0  40  100  17.773892   93.083333   90.0  24  150   
water     70.219298   68.0  20  170  27.141960   73.307018   70.0  10  155   

                     
                std  
type1                
bug       37.202884  
dark      25.470402  
dragon    33.864276  
electric  25.502190  
fairy     28.682622  
fighting  27.966983  
fire      26.895951  
flying    35.118846  
ghost     32.577237  
grass     29.880682  
ground    31.467789  
ice       29.355865  
normal    30.749022  
poison    19.871538  
psychic   36.842445  
rock      35.180702  
steel     29.525106  
water     28.883081

In [29]:
# Reset row index
result_reset = result.reset_index()
result_reset

type1         hp                                 attack             \
                   mean median min  max        std        mean median min   
0        bug  56.722222   60.0   1  107  17.652532   70.125000   65.0  10   
1       dark  72.551724   65.0  35  223  36.063026   87.793103   88.0  50   
2     dragon  79.851852   75.0  41  216  34.962291  106.407407  100.0  50   
3   electric  60.512821   60.0  20   90  17.519721   70.820513   65.0  30   
4      fairy  73.944444   76.0  35  126  23.048471   62.111111   58.5  20   
5   fighting  71.428571   70.0  30  144  26.790585   99.178571  100.0  35   
6       fire  68.730769   65.0  38  115  19.182422   81.500000   79.5  30   
7     flying  68.000000   79.0  40   85  24.433583   66.666667   70.0  30   
8      ghost  63.370370   59.0  20  150  30.011726   72.740741   66.0  30   
9      grass  65.358974   65.0  30  123  18.721454   73.769231   68.0  27   
10    ground  73.187500   72.5  10  115  25.518415   94.812500   90.0  40   
11       ice  72.086957   70.0  36  110  21.760918   73.304348   65.0  30   
12    normal  76.723810   70.0  30  255  35.460838   75.161905   75.0   5   
13    poison  65.593750   65.0  35  105  19.108956   72.656250   68.5  43   
14   psychic  72.943396   68.0  20  190  31.507884   65.566038   55.0  20   
15      rock  66.333333   67.0  30  123  21.781978   90.666667   89.0  40   
16     steel  66.791667   60.0  40  100  17.773892   93.083333   90.0  24   
17     water  70.219298   68.0  20  170  27.141960   73.307018   70.0  10   

                    
    max        std  
0   185  37.202884  
1   150  25.470402  
2   180  33.864276  
3   123  25.502190  
4   131  28.682622  
5   145  27.966983  
6   160  26.895951  
7   100  35.118846  
8   165  32.577237  
9   181  29.880682  
10  180  31.467789  
11  130  29.355865  
12  160  30.749022  
13  106  19.871538  
14  165  36.842445  
15  165  35.180702  
16  150  29.525106  
17  155  28.883081

In [30]:
# Inspect the multi-level column labels
print(f"{result.columns=}")
print(f"{result_reset.columns=}")

result.columns=MultiIndex([(    'hp',   'mean'),
            (    'hp', 'median'),
            (    'hp',    'min'),
            (    'hp',    'max'),
            (    'hp',    'std'),
            ('attack',   'mean'),
            ('attack', 'median'),
            ('attack',    'min'),
            ('attack',    'max'),
            ('attack',    'std')],
           )
result_reset.columns=MultiIndex([( 'type1',       ''),
            (    'hp',   'mean'),
            (    'hp', 'median'),
            (    'hp',    'min'),
            (    'hp',    'max'),
            (    'hp',    'std'),
            ('attack',   'mean'),
            ('attack', 'median'),
            ('attack',    'min'),
            ('attack',    'max'),
            ('attack',    'std')],
           )


In [31]:
# Access level-0 column
# Leaving the row labels of a pandas DataFrame as the group name is more convenient here
# Because the label always stays with the row, so we don’t need to keep an extra column like 'type1' to identify it explicitly

# Row labels remain the group name ('type1')
display( result['hp'] )         # This equals result.loc[ :, 'hp' ]

# Row labels are reset to default values
display( result_reset['hp'] )   # This equals result_reset.loc[ :, 'hp' ]
display( result_reset.loc[ :, ['type1','hp'] ] )

,mean,median,min,max,std
type1,,,,,
bug,56.722222,60.0,1,107,17.652532
dark,72.551724,65.0,35,223,36.063026
dragon,79.851852,75.0,41,216,34.962291
electric,60.512821,60.0,20,90,17.519721
fairy,73.944444,76.0,35,126,23.048471
fighting,71.428571,70.0,30,144,26.790585
fire,68.730769,65.0,38,115,19.182422
flying,68.000000,79.0,40,85,24.433583
ghost,63.370370,59.0,20,150,30.011726


,mean,median,min,max,std
0,56.722222,60.0,1,107,17.652532
1,72.551724,65.0,35,223,36.063026
2,79.851852,75.0,41,216,34.962291
3,60.512821,60.0,20,90,17.519721
4,73.944444,76.0,35,126,23.048471
5,71.428571,70.0,30,144,26.790585
6,68.730769,65.0,38,115,19.182422
7,68.000000,79.0,40,85,24.433583
8,63.370370,59.0,20,150,30.011726
9,65.358974,65.0,30,123,18.721454


type1         hp                           
                   mean median min  max        std
0        bug  56.722222   60.0   1  107  17.652532
1       dark  72.551724   65.0  35  223  36.063026
2     dragon  79.851852   75.0  41  216  34.962291
3   electric  60.512821   60.0  20   90  17.519721
4      fairy  73.944444   76.0  35  126  23.048471
5   fighting  71.428571   70.0  30  144  26.790585
6       fire  68.730769   65.0  38  115  19.182422
7     flying  68.000000   79.0  40   85  24.433583
8      ghost  63.370370   59.0  20  150  30.011726
9      grass  65.358974   65.0  30  123  18.721454
10    ground  73.187500   72.5  10  115  25.518415
11       ice  72.086957   70.0  36  110  21.760918
12    normal  76.723810   70.0  30  255  35.460838
13    poison  65.593750   65.0  35  105  19.108956
14   psychic  72.943396   68.0  20  190  31.507884
15      rock  66.333333   67.0  30  123  21.781978
16     steel  66.791667   60.0  40  100  17.773892
17     water  70.219298   68.0  20  170  27.141960

In [32]:
# Access level-1 column
# Leaving the row labels of a pandas DataFrame as the group name is more convenient here
# Because the label always stays with the row, so we don’t need to keep an extra column like 'type1' to identify it explicitly

# Row labels remain the group name ('type1')
display( result['hp']['mean'] )         # This equals result.loc[ : , ('hp','mean') ]

# Row labels are reset to default values
display( result_reset['hp']['mean'] )   # This equals result_reset.loc[ : , ('hp','mean') ]
display( result_reset.loc[ :, [('type1',''),('hp','mean')] ] )

type1
bug         56.722222
dark        72.551724
dragon      79.851852
electric    60.512821
fairy       73.944444
fighting    71.428571
fire        68.730769
flying      68.000000
ghost       63.370370
grass       65.358974
ground      73.187500
ice         72.086957
normal      76.723810
poison      65.593750
psychic     72.943396
rock        66.333333
steel       66.791667
water       70.219298
Name: mean, dtype: float64

0     56.722222
1     72.551724
2     79.851852
3     60.512821
4     73.944444
5     71.428571
6     68.730769
7     68.000000
8     63.370370
9     65.358974
10    73.187500
11    72.086957
12    76.723810
13    65.593750
14    72.943396
15    66.333333
16    66.791667
17    70.219298
Name: mean, dtype: float64

,type1,hp
,,mean
0,bug,56.722222
1,dark,72.551724
2,dragon,79.851852
3,electric,60.512821
4,fairy,73.944444
5,fighting,71.428571
6,fire,68.730769
7,flying,68.000000
8,ghost,63.370370


### 3.2.2 Split $→$ Transform

**EX1:** Standardization (this function has no aggregation effect)

## Notice

>หลังทำการ groupby() แล้วเอามา aggregate โดยใช้ transform จะได้ จำนวน row เท่าเดิม ที่มีค่า aggregate ซ้ำๆกัน เพิ่มมาอีก column

>ไม่มีการยุบข้อมูล   

>ต้องทำ groupby ก่อน ค่อยเอามา aggregate โดยใช้t transform โดยมีผลลัพธ์นั้นแปะอยู่ใน column จำนวน row เท่าเดิม

While `groupby()`+aggregation (<a href='#sect3-2-1'>Section 3.2.1</a>: Split $→$ Aggregate) and `groupby()`+`apply(aggregate_func)` return a reduced (grouped) version of the input, `groupby()`+`transform(aggregate_func)` returns a group-level transformed version <u>of the full data</u>. The new output data has the same length (the same number of rows) as the input data.

![picture](https://miro.medium.com/max/826/1*fxdhr6i-jlzeex7f5RkIlw.png)

Image from <a href="https://iamirmasoud.com/2021/06/26/a-complete-guide-on-pandas-grouping-aggregating-and-transformation/">https://iamirmasoud.com/2021/06/26/a-complete-guide-on-pandas-grouping-aggregating-and-transformation/</a>

In [33]:
# Create a pandas.DataFrame to play around
df = df_pokemon[ ['pokedex_number','name','type1','type2','hp','attack','defense'] ]

In [ ]:
# Before standardization: Inspect mean and SD values of all groups

# Group the data
# Explanation about 'group_keys' in the following EX2
grouped = df.groupby('type1', group_keys=False)

# Check the current mean and SD for each group in 'type1'
print( f"{'='*10} Group by 'type1' -> Find mean {'='*10}\n" )
print( grouped.mean(numeric_only=True), end='\n\n' )
print( f"{'='*10} Group by 'type1' -> Find SD {'='*10}\n" )
print( grouped.std(numeric_only=True) )

In [ ]:
# Perform standardization to each group of 'type1'

# Group -> Choose columns -> Data transformation
# Two commands below produce the same results, use only one
#transformed = grouped[['hp','attack','defense']].apply( lambda x: ((x - x.mean()) / x.std()) )
transformed = grouped[['hp','attack','defense']].transform( lambda x: ((x - x.mean()) / x.std()) )

# Notice that results below are no longer grouped by 'type1'
transformed

In [ ]:
# To easily compare values, assign the transformed result to new columns
df2 = df.assign(hp_zscore=transformed['hp'], \
                att_zscore=transformed['attack'], \
                def_zscore=transformed['defense'])

df2

In [ ]:
# After standardization: Reconfirm mean and sd values of each group

# Notice that mean and SD values of each group are about 0.0 and 1.0 respectively
print( f"{'='*5} Group by 'type1' -> Standardize -> Mean {'='*5}\n" )
print( df2.groupby('type1')[['hp_zscore','att_zscore','def_zscore']].mean() )
print( f"\n{'='*5} Group by 'type1' -> Standardize -> SD {'='*5}\n" )
print( df2.groupby('type1')[['hp_zscore','att_zscore','def_zscore']].std() )

**EX2:** Difference between `apply()` and `transform()` when using with groupby

In [ ]:
# Prepare the DataFrame to play around
df = df_pokemon[ ['pokedex_number','name','type1','hp','attack','defense'] ]
df.head()

When the function IS NOT an aggregate function:

In [ ]:
# The 'group_keys' argument when using df.groupby(..., group_keys).apply(...)
# - When 'group_keys=True', the group labels are prepended to the resultant DataFrame
# - When 'group_keys=False', the group labels are omitted in the resultant DataFrame
# The 'group_keys' argument only matters when the result from df.groupby().apply() has the same shape as the original 'df'

print(f'************ group_keys=False ************')
df_false = df.groupby('type1', group_keys=False)[['hp','attack','defense']].apply(lambda x: x/100)
print( df_false.index, end='\n\n' )
display( df_false )

print(f'\n************ group_keys=True ************')
df_true = df.groupby('type1', group_keys=True)[['hp','attack','defense']].apply(lambda x: x/100)
print( df_true.index, end='\n\n' )
display( df_true )

In [ ]:
# Groupby -> Choose columns -> Use a non-aggregate function

# These two commands produce the same result (DataFrame), use only one
# Notice that the results are no longer grouped by type1
#df.groupby('type1', group_keys=False)[['hp','attack','defense']].apply(lambda x: x/100)
df.groupby('type1')[['hp','attack','defense']].transform(lambda x: x/100)

In [ ]:
# Groupby -> Choose columns -> Use a non-aggregate function
# Remind that apply() sends the whole DataFrame to the calling function whereas transform() only sends each column separately

# OK
# Notice that the results are no longer grouped by 'type1'
df.groupby('type1', group_keys=False)[['hp','attack','defense']] \
                                        .apply(lambda x: (x['attack'] - x['defense']).to_frame('att_sub_def') )

# KeyError
#df.groupby('type1')[['hp','attack','defense']].transform(lambda x: (x['attack'] - x['defense']).to_frame('att_sub_def') )

When the function IS an aggregate function:

In [ ]:
# Groupby -> Choose columns -> Use an aggregate function via apply()
# When the result from .apply() doesn't retain the shape of 'df', 'group_keys' has no effect

# When the function includes aggregation, apply() returns one value per one group
df.groupby('type1')[['hp','attack','defense']].apply(lambda x: x.max())

In [ ]:
# Groupby -> Choose columns -> Use an aggregate function via transform() -> Append the result to the original DataFrame
# Even when the function includes aggregation, transform() returns results with the same length as the input

# Transform the groupby object
transformed = df.groupby('type1')[['hp','attack','defense']].transform(lambda x: x.max())
print('******* DataFrame resulted from .transform() *******')
display( transformed )

# Append the tranform results to the original DataFrame
print('\n******* Original DataFrame with transformed columns appended *******')
df2 = df.assign( type1_hp_max=transformed.hp, type1_att_max=transformed.attack, type1_def_max=transformed.defense)
df2

******* DataFrame resulted from .transform() ******* /n
******* Original DataFrame with transformed columns appended *******

### 3.2.3 Split $→$ Filtration

- Filtration is a process in which we <u>discard some groups</u>, according to a group-wise computation that evaluates True or False.
- As the name suggests, `filter()` <u>does not change the data in any capacity, but instead selects a subset of the data</u>.

![picture](https://miro.medium.com/max/826/1*lDZgurT8zja-u77DW758KQ.png)

Image from <a href="https://iamirmasoud.com/2021/06/26/a-complete-guide-on-pandas-grouping-aggregating-and-transformation/">https://iamirmasoud.com/2021/06/26/a-complete-guide-on-pandas-grouping-aggregating-and-transformation/</a>

## Filter เหมือน having ใน SQL

In [34]:
# Create a DataFrame to play around
df = df_pokemon[ ['pokedex_number','name','type1','hp','attack','defense'] ]
df.head()

,pokedex_number,name,type1,hp,attack,defense
0,1,Bulbasaur,grass,45,49,49
1,2,Ivysaur,grass,60,62,63
2,3,Venusaur,grass,80,100,123
3,4,Charmander,fire,39,52,43
4,5,Charmeleon,fire,58,64,58


In [35]:
# Groupby without filter
df.groupby('type1').size()

type1
bug          72
dark         29
dragon       27
electric     39
fairy        18
fighting     28
fire         52
flying        3
ghost        27
grass        78
ground       32
ice          23
normal      105
poison       32
psychic      53
rock         45
steel        24
water       114
dtype: int64

In [36]:
# Groupby with a filter

# Groupby -> Filtration
# Show only data that belongs to a group whose number of pokemons is less than 20
result = df.groupby('type1').filter(lambda group: group['pokedex_number'].count() < 20)

# Result from groupby+filter is the subset of the original dataframe (df) not the grouped results
result

,pokedex_number,name,type1,hp,attack,defense
34,35,Clefairy,fairy,70,45,48
35,36,Clefable,fairy,95,70,73
172,173,Cleffa,fairy,50,25,28
174,175,Togepi,fairy,35,20,65
175,176,Togetic,fairy,55,40,85
208,209,Snubbull,fairy,60,80,50
209,210,Granbull,fairy,90,120,75
467,468,Togekiss,fairy,85,50,95
640,641,Tornadus,flying,79,100,80
668,669,Flabébé,fairy,44,38,39


## 3.3 Groupby's wrapup
1. Result returned from `.groupby()` is neither pandas.Series nor pandas.DataFrame, but a Groupby object.

2. A Groupby object can be further processed with aggregation functions. Result is <u>an aggregated DataFrame whose number of rows equal to the number of groups</u>.
    - Standalone aggregation function: `df.groupby(...).mean(...)`, ...
    - Compound aggregation function: `df.groupby(...).agg(...)`

3. A Groupby object can be further processed with apply functions, `df.groupby(..., group_keys).apply(func)`:
    - `func` can be any function with-or-without aggregation effect and with-or-without cross-colomn calculation.
    - If `func` is a function with no aggregation effect, result is <u>a non-aggregated DataFrame whose number of rows equal to the original DataFrame before being grouped</u>. In this case, watch out the bool value in `group_keys`.
    - If `func` is a function with aggregation effect, result is <u>an aggregated DataFrame whose number of rows equal to the number of groups</u>.

4. A Groupby object can be further processed with tranformation functions, `df.groupby(...).tranform(func)`:
    - `func` can be any function with-or-without aggregation effect but <u>any cross-colomn calculation will cause an error</u>.
    - If `func` is a function with no aggregation effect, it is similar to calling `df.groupby(..., group_keys=False).apply(func)`. Result is <u>a non-aggregated DataFrame whose number of rows equal to the original DataFrame before being grouped</u>.
    - If `func` is a function with aggregation effect, result is <u>seemingly a non-aggregated DataFrame whose number of rows equal to the number of groups</u>. However, the data in transformed columns are not the same as the data in the original DataFrame (before being grouped) <u>but they are the per-group aggregated results.</u>

5. A Groupby object can be further processed with filtration functions, `df.groupby(...).filter(group_func)`:
    - `group_func` is a function that receives one group as input and returns True/False (= Does this group meet our criteria?).
    - Result from `df.groupby(...).filter(group_func)` is a direct subset of the original DataFrame before being grouped (no data in `df` is changed). <u>However, some rows in `df` will be filtered out (eliminated) if they belong to groups that don't meet our criteria judging by `group_func`</u>.

# PRACTICE

1. (1 point) Based on the 'type1' attribute, conclude the numbers of each pokemon type. Sort the results descendingly based on the numbers of pokemons.

2. (1 point) Display all types of pokemons (considering 'type1' only) from the best to the worst stats.
    - First, consider 'hp', the larger the better. Then, consider the average between 'attack' and 'defense', the larger the better
    - The final display should include at least 'type1', 'hp', 'attack' and 'defense' columns.

3. (1.5 point) Display all pokemons whose 'hp' is greater than the average hp value computed from all pokemons with the same 'type1'. Sort the result by 'type1' (ascendingly) then by 'hp' (descendingly).